### imports the web requests package, establishes your working directory path, and defines the exact data codes used by the World Bank for Singapore.

In [1]:
import os
import requests
import pandas as pd
import numpy as np

# Create path for storing your raw downloads
RAW_DATA_PATH = "../data/raw"
os.makedirs(RAW_DATA_PATH, exist_ok=True)

# Country code for Singapore
COUNTRY_CODE = "SG"

# Exact World Bank API indicator IDs mapped to descriptive labels
INDICATORS = {
    "NY.GDP.MKTP.CD": "gdp_nominal_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_annual_pct",
    "FP.CPI.TOTL.ZG": "inflation_rate_pct",
    "NE.EXP.GNFS.ZS": "exports_pct_gdp",
    "GC.XPN.TOTL.GD.ZS": "gov_expense_pct_gdp",
    "SP.POP.TOTL": "total_population",
    "SP.POP.1564.TO.ZS": "working_age_pop_pct",
    "SE.XPD.TOTL.GD.ZS": "education_expenditure_pct_gdp"
}


The World Bank API wraps its data inside a nested JSON structure where the first item contains metadata (like pagination details) and the second item contains the actual timeline records. This function clean-parses that logic safely.

In [2]:
def fetch_world_bank_data(country, indicators, start_year=1998, end_year=2025):
    """
    Programmatically loops through indicators, queries the World Bank API,
    and merges the results into a single comprehensive DataFrame.
    """
    master_df = pd.DataFrame()
    
    for code, label in indicators.items():
        print(f"Requesting data for: {label} ({code})...")
        
        # World Bank API REST endpoint structure
        url = f"http://worldbank.org{country}/indicator/{code}"
        params = {
            "date": f"{start_year}:{end_year}",
            "format": "json",
            "per_page": 1000  # Large page size prevents pagination breaks
        }
        
        try:
            response = requests.get(url, params=params)
            
            if response.status_code == 200:
                data = response.json()
                
                # Check if the response contains valid record data
                if len(data) > 1 and data[1] is not None:
                    records = data[1]
                    
                    # Extract only the year and value from the nested json response
                    parsed_records = []
                    for item in records:
                        if item["value"] is not None:
                            parsed_records.append({
                                "year": int(item["date"]),
                                label: float(item["value"])
                            })
                    
                    # Convert to dataframe
                    df = pd.DataFrame(parsed_records)
                    
                    # Merge into our master timeline dataframe
                    if master_df.empty:
                        master_df = df
                    else:
                        master_df = pd.merge(master_df, df, on="year", how="outer")
                else:
                    print(f"[-] No data found for indicator: {label}")
            else:
                print(f"[-] HTTP Error {response.status_code} for {label}")
                
        except Exception as e:
            print(f"[-] Network connection error fetching {label}: {e}")
            
    # Sort timeline oldest to newest
    if not master_df.empty:
        master_df = master_df.sort_values("year").reset_index(drop=True)
        
    return master_df


#### execute your logic. It will download the data directly into your local workspace.

In [3]:
# Execute the API pipeline
sg_worldbank_df = fetch_world_bank_data(COUNTRY_CODE, INDICATORS)

# View data structural results
print("\n--- Data Ingestion Complete ---")
print(f"Data Matrix Shape: {sg_worldbank_df.shape} (Rows, Columns)")
print("\nRecent Data Matrix Sample:")
print(sg_worldbank_df.tail())

# Save dataset to your git-ignored raw data path
output_file = os.path.join(RAW_DATA_PATH, "singapore_worldbank_raw.csv")
sg_worldbank_df.to_csv(output_file, index=False)
print(f"\n[SUCCESS] Raw dataset cached locally to: {output_file}")


Requesting data for: gdp_nominal_usd (NY.GDP.MKTP.CD)...
[-] Network connection error fetching gdp_nominal_usd: HTTPConnectionPool(host='worldbank.orgsg', port=80): Max retries exceeded with url: /indicator/NY.GDP.MKTP.CD?date=1998%3A2025&format=json&per_page=1000 (Caused by NameResolutionError("HTTPConnection(host='worldbank.orgsg', port=80): Failed to resolve 'worldbank.orgsg' ([Errno 11001] getaddrinfo failed)"))
Requesting data for: gdp_growth_annual_pct (NY.GDP.MKTP.KD.ZG)...
[-] Network connection error fetching gdp_growth_annual_pct: HTTPConnectionPool(host='worldbank.orgsg', port=80): Max retries exceeded with url: /indicator/NY.GDP.MKTP.KD.ZG?date=1998%3A2025&format=json&per_page=1000 (Caused by NameResolutionError("HTTPConnection(host='worldbank.orgsg', port=80): Failed to resolve 'worldbank.orgsg' ([Errno 11001] getaddrinfo failed)"))
Requesting data for: inflation_rate_pct (FP.CPI.TOTL.ZG)...
[-] Network connection error fetching inflation_rate_pct: HTTPConnectionPool(host=